In [0]:
import sys, os

sys.path.append("../log")
try:
    from audit_logging import log_pipeline_execution
except ModuleNotFoundError:
    nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    log_dir = os.path.normpath("/Workspace" + os.path.dirname(nb_path) + "/../log")
    sys.path.append(log_dir)
    from audit_logging import log_pipeline_execution

print("Audit helper loaded:", log_pipeline_execution.__name__)

In [0]:
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window

BRONZE            = "workspace.tmdb.bronze_movies"
SILVER_MOVIES     = "workspace.tmdb.silver_movies"
SILVER_TRENDING   = "workspace.tmdb.silver_movie_trending"
SILVER_QUARANTINE = "workspace.tmdb.silver_quarantine"
SILVER_DQ         = "workspace.tmdb.silver_dq_results"

dbutils.widgets.text("bronze_batch_id", "", "Bronze batch to process (blank = all batches)")
bronze_batch = dbutils.widgets.get("bronze_batch_id").strip()
run_id = datetime.now().strftime("S%Y%m%d_%H%M%S")
print(f"Silver run_id={run_id} | bronze_batch='{bronze_batch or 'ALL'}'")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_MOVIES} (
    tmdb_id                BIGINT         NOT NULL,
    movie_nk               STRING         NOT NULL,
    movie_title            STRING         NOT NULL,
    original_title         STRING,
    release_date           DATE,
    release_year           INT,
    running_time_min       INT,
    release_status         STRING         NOT NULL,
    primary_genre          STRING,
    franchise              STRING         NOT NULL,
    primary_country        STRING,
    primary_language       STRING,
    primary_distributor    STRING         NOT NULL,
    distributor_id         STRING         NOT NULL,
    production_budget_usd  DECIMAL(14,2),
    revenue_usd            DECIMAL(14,2),
    vote_average           DOUBLE,
    vote_count             INT,
    popularity             DOUBLE,
    imdb_id                STRING,
    is_adult               BOOLEAN,
    source_collection      STRING,
    pulled_at              TIMESTAMP,
    bronze_batch_id        STRING,
    silver_run_id          STRING,
    silver_load_ts         TIMESTAMP
) USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TRENDING} (
    tmdb_id          BIGINT NOT NULL,
    snapshot_date    DATE   NOT NULL,
    popularity       DOUBLE,
    vote_average     DOUBLE,
    vote_count       INT,
    pulled_at        TIMESTAMP,
    bronze_batch_id  STRING,
    silver_run_id    STRING,
    silver_load_ts   TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_QUARANTINE} (
    bronze_batch_id   STRING,
    source_collection STRING,
    source_file       STRING,
    raw_record        STRING,
    reject_reason     STRING,
    silver_run_id     STRING,
    quarantined_at    TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_DQ} (
    silver_run_id STRING,
    category      STRING,
    check_name    STRING,
    table_name    STRING,
    failed_rows   BIGINT,
    total_rows    BIGINT,
    severity      STRING,
    status        STRING,
    checked_at    TIMESTAMP
) USING DELTA
""")

In [0]:
MOVIE_SCHEMA = """
  id STRING, title STRING, original_title STRING, release_date STRING, status STRING,
  budget STRING, revenue STRING, runtime STRING, popularity STRING, vote_average STRING, vote_count STRING,
  original_language STRING, imdb_id STRING, adult BOOLEAN,
  belongs_to_collection STRUCT<id: STRING, name: STRING>,
  genres ARRAY<STRUCT<id: STRING, name: STRING>>,
  production_companies ARRAY<STRUCT<id: STRING, name: STRING>>,
  production_countries ARRAY<STRUCT<iso_3166_1: STRING, name: STRING>>,
  spoken_languages ARRAY<STRUCT<iso_639_1: STRING, english_name: STRING>>
"""

def clean(c):
    """trim + collapse inner whitespace; empty string -> NULL"""
    x = F.trim(F.regexp_replace(c, r"\s+", " "))
    return F.when(x == "", F.lit(None)).otherwise(x)

def tc(expr_str):
    return F.expr(expr_str)

In [0]:
start_time = datetime.now()
status, err = "SUCCESS", None
rows_inserted = rows_updated = rows_quarantined = 0

bronze = spark.table(BRONZE)
if bronze_batch:
    bronze = bronze.filter(F.col("batch_id") == bronze_batch)

parsed = (bronze
    .withColumn("j", F.from_json("raw_record", MOVIE_SCHEMA))
    .withColumn("pulled_ts", F.coalesce(F.expr("try_to_timestamp(pulled_at)"), F.col("load_timestamp"))))

p = parsed.select(
    F.col("batch_id").alias("bronze_batch_id"), "source_collection", "source_file", "raw_record",
    "load_timestamp", "pulled_ts", "j",
    tc("try_cast(j.id AS BIGINT)").alias("tmdb_id"),
    clean(F.col("j.title")).alias("movie_title"),
).withColumn(
    "reject_reason",
    F.when(F.col("j").isNull(), "BAD_JSON")
     .when(F.col("tmdb_id").isNull(), "NULL_ID")
     .when(F.col("movie_title").isNull(), "NULL_TITLE")
).cache()

rejects = (p.filter("reject_reason IS NOT NULL")
            .select("bronze_batch_id", "source_collection", "source_file", "raw_record", "reject_reason")
            .withColumn("silver_run_id", F.lit(run_id))
            .withColumn("quarantined_at", F.current_timestamp())
            .dropDuplicates(["bronze_batch_id", "source_file", "raw_record", "reject_reason"]))
rows_quarantined = rejects.count()

(DeltaTable.forName(spark, SILVER_QUARANTINE).alias("t")
    .merge(rejects.alias("s"),
           "t.bronze_batch_id = s.bronze_batch_id AND t.source_file = s.source_file "
           "AND t.raw_record = s.raw_record AND t.reject_reason = s.reject_reason")
    .whenNotMatchedInsertAll().execute())

good = p.filter("reject_reason IS NULL")
print(f"Good records: {good.count()} | Quarantined: {rows_quarantined}")

In [0]:
details = good.filter(F.col("source_collection").isin("full_load", "changed"))

rel_date = tc("try_cast(try_to_timestamp(j.release_date, 'yyyy-MM-dd') AS DATE)")
budget   = tc("try_cast(j.budget  AS DECIMAL(14,2))")
revenue  = tc("try_cast(j.revenue AS DECIMAL(14,2))")
runtime  = tc("try_cast(j.runtime AS INT)")
dist_nm  = clean(tc("try_element_at(j.production_companies, 1).name"))

m = (details
    .withColumn("release_date", rel_date)
    .withColumn("dist_name", dist_nm)
    .select(
        "tmdb_id", "movie_title", "bronze_batch_id", "source_collection", "pulled_ts", "load_timestamp",
        clean(F.col("j.original_title")).alias("original_title"),
        "release_date",
        F.year("release_date").alias("release_year"),
        F.when(runtime.between(1, 600), runtime).alias("running_time_min"),
        F.when((F.col("release_date") <= F.current_date()) | (F.col("j.status") == "Released"), "Released")
         .otherwise("Upcoming").alias("release_status"),
        F.initcap(clean(tc("try_element_at(j.genres, 1).name"))).alias("primary_genre"),
        F.coalesce(clean(F.regexp_replace(F.col("j.belongs_to_collection.name"), r"\s+Collection$", "")),
                   F.lit("No Franchise")).alias("franchise"),
        F.initcap(clean(tc("try_element_at(j.production_countries, 1).name"))).alias("primary_country"),
        F.initcap(F.coalesce(clean(tc("try_element_at(j.spoken_languages, 1).english_name")),
                             clean(F.col("j.original_language")))).alias("primary_language"),
        F.coalesce(F.col("dist_name"), F.lit("Unknown")).alias("primary_distributor"),
        F.coalesce(
            F.upper(F.regexp_replace(F.regexp_replace(F.lower("dist_name"), r"[^a-z0-9]+", "_"), r"^_|_$", "")),
            F.lit("UNKNOWN")).alias("distributor_id"),
        F.when(budget > 0, budget).alias("production_budget_usd"),
        F.when(revenue > 0, revenue).alias("revenue_usd"),
        tc("try_cast(j.vote_average AS DOUBLE)").alias("vote_average"),
        tc("try_cast(j.vote_count AS INT)").alias("vote_count"),
        tc("try_cast(j.popularity AS DOUBLE)").alias("popularity"),
        clean(F.col("j.imdb_id")).alias("imdb_id"),
        F.col("j.adult").alias("is_adult"),
    )
    .withColumn("movie_nk", F.concat_ws("|", F.lower("movie_title"),
                                        F.coalesce(F.col("release_date").cast("string"), F.lit("unknown"))))
)

# keep the freshest version per movie; 'changed' beats 'full_load' on a tie
w = Window.partitionBy("tmdb_id").orderBy(
    F.col("pulled_ts").desc(),
    F.when(F.col("source_collection") == "changed", 1).otherwise(2),
    F.col("load_timestamp").desc())

movies = (m.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "load_timestamp")
           .withColumnRenamed("pulled_ts", "pulled_at")
           .withColumn("silver_run_id", F.lit(run_id))
           .withColumn("silver_load_ts", F.current_timestamp()))

new_ids   = movies.join(spark.table(SILVER_MOVIES).select("tmdb_id"), "tmdb_id", "left_anti").count()
total_src = movies.count()

(DeltaTable.forName(spark, SILVER_MOVIES).alias("t")
    .merge(movies.alias("s"), "t.tmdb_id = s.tmdb_id")
    .whenMatchedUpdateAll(condition="s.pulled_at >= t.pulled_at")
    .whenNotMatchedInsertAll().execute())

rows_inserted, rows_updated = new_ids, total_src - new_ids
print(f"silver_movies | new={new_ids} updated={rows_updated}")

In [0]:
t = (good.filter(F.col("source_collection") == "trending")
    .select("tmdb_id", "bronze_batch_id", "pulled_ts",
            F.to_date("pulled_ts").alias("snapshot_date"),
            tc("try_cast(j.popularity AS DOUBLE)").alias("popularity"),
            tc("try_cast(j.vote_average AS DOUBLE)").alias("vote_average"),
            tc("try_cast(j.vote_count AS INT)").alias("vote_count")))

wt = Window.partitionBy("tmdb_id", "snapshot_date").orderBy(F.col("pulled_ts").desc())
trending = (t.withColumn("_rn", F.row_number().over(wt)).filter("_rn = 1").drop("_rn")
             .withColumnRenamed("pulled_ts", "pulled_at")
             .withColumn("silver_run_id", F.lit(run_id))
             .withColumn("silver_load_ts", F.current_timestamp()))

(DeltaTable.forName(spark, SILVER_TRENDING).alias("t")
    .merge(trending.alias("s"), "t.tmdb_id = s.tmdb_id AND t.snapshot_date = s.snapshot_date")
    .whenMatchedUpdateAll(condition="s.pulled_at >= t.pulled_at")
    .whenNotMatchedInsertAll().execute())

rows_inserted += trending.count()   # approximate
p.unpersist()

# Audit log (assumes error_message is accepted; falls back if not)
kwargs = dict(layer="Bronze-to-Silver", load_type="all", batch_id=run_id,
              parameter=bronze_batch or "ALL", start_time=start_time, end_time=datetime.now(),
              status="SUCCESS", rows_inserted=rows_inserted, rows_updated=rows_updated,
              rows_quarantined=rows_quarantined)
try:
    log_pipeline_execution(**kwargs, error_message=None)
except TypeError:
    log_pipeline_execution(**kwargs)

print("Silver load complete")

In [0]:
mv, tr = spark.table(SILVER_MOVIES), spark.table(SILVER_TRENDING)
n_mv, n_tr = mv.count(), tr.count()

def chk(category, name, table, failed, total, severity="ERROR"):
    st = ("PASS" if failed == 0 else "FAIL") if severity == "ERROR" else "INFO"
    return (run_id, category, name, table, int(failed), int(total), severity, st, datetime.now())

orphans    = tr.join(mv, "tmdb_id", "left_anti").select("tmdb_id").distinct().count()
dup_movies = mv.groupBy("tmdb_id").count().filter("count > 1").count()
dup_trend  = tr.groupBy("tmdb_id", "snapshot_date").count().filter("count > 1").count()

results = [
    # Completeness
    chk("Completeness", "movie_title not null", "silver_movies", mv.filter("movie_title IS NULL").count(), n_mv),
    chk("Completeness", "release_date missing", "silver_movies", mv.filter("release_date IS NULL").count(), n_mv, "INFO"),
    chk("Completeness", "running_time_min missing", "silver_movies", mv.filter("running_time_min IS NULL").count(), n_mv, "INFO"),
    chk("Completeness", "production_budget_usd missing", "silver_movies", mv.filter("production_budget_usd IS NULL").count(), n_mv, "INFO"),
    chk("Completeness", "revenue_usd missing", "silver_movies", mv.filter("revenue_usd IS NULL").count(), n_mv, "INFO"),
    chk("Completeness", "primary_genre missing", "silver_movies", mv.filter("primary_genre IS NULL").count(), n_mv, "INFO"),
    # Accuracy
    chk("Accuracy", "budget/revenue are positive", "silver_movies", mv.filter("production_budget_usd < 0 OR revenue_usd < 0").count(), n_mv),
    chk("Accuracy", "runtime within 1-600 min", "silver_movies", mv.filter("running_time_min NOT BETWEEN 1 AND 600").count(), n_mv),
    chk("Accuracy", "release_date plausible", "silver_movies",
        mv.filter("release_date < DATE'1888-01-01' OR release_date > add_months(current_date(), 60)").count(), n_mv),
    chk("Accuracy", "vote_average within 0-10", "silver_movies", mv.filter("vote_average NOT BETWEEN 0 AND 10").count(), n_mv),
    chk("Accuracy", "no edge/double spaces in title", "silver_movies",
        mv.filter("movie_title <> trim(movie_title) OR movie_title RLIKE '\\\\s{2,}'").count(), n_mv),
    # Integrity
    chk("Integrity", "tmdb_id unique", "silver_movies", dup_movies, n_mv),
    chk("Integrity", "(tmdb_id, snapshot_date) unique", "silver_movie_trending", dup_trend, n_tr),
    chk("Integrity", "trending movies exist in silver_movies", "silver_movie_trending", orphans,
        tr.select("tmdb_id").distinct().count(), "INFO"),
    chk("Integrity", "distributor_id never null", "silver_movies", mv.filter("distributor_id IS NULL").count(), n_mv),
]

# Reconciliation: only meaningful when the whole bronze table was processed
if not bronze_batch:
    src_ids = (spark.table(BRONZE).filter("source_collection IN ('full_load','changed')")
               .select(F.expr("try_cast(get_json_object(raw_record,'$.id') AS BIGINT)").alias("id"))
               .filter("id IS NOT NULL").distinct().count())
    results.append(chk("Reconciliation", "bronze detail ids = silver_movies rows",
                       "silver_movies", abs(src_ids - n_mv), src_ids))

dq = spark.createDataFrame(results,
    "silver_run_id string, category string, check_name string, table_name string, "
    "failed_rows long, total_rows long, severity string, status string, checked_at timestamp")
dq.write.mode("append").saveAsTable(SILVER_DQ)
display(dq.orderBy("category", "check_name"))

In [0]:
display(spark.sql(f"""
  SELECT 'silver_movies' AS tbl, COUNT(*) AS n FROM {SILVER_MOVIES}
  UNION ALL SELECT 'silver_movie_trending', COUNT(*) FROM {SILVER_TRENDING}
  UNION ALL SELECT 'silver_quarantine',     COUNT(*) FROM {SILVER_QUARANTINE}
"""))
display(spark.sql(f"SELECT reject_reason, COUNT(*) n FROM {SILVER_QUARANTINE} GROUP BY reject_reason"))
display(spark.table(SILVER_MOVIES).limit(20))